In [12]:
import os
import re
import logging
import fitz  # PyMuPDF
import pandas as pd
import spacy
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed
from PIL import Image
import pytesseract
from collections import defaultdict
import time
from tqdm import tqdm
import gc

# Configure logging
logging.basicConfig(
    filename='extraction.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# Global constants
NLP_MODEL = "en_core_web_lg"
MIN_SENTENCE_LENGTH = 15  # Filter out fragments
MAX_SENTENCE_LENGTH = 500  # Prevent paragraph captures
CONTEXT_WINDOW = 2  # Sentences before/after for context
BATCH_SIZE = 50  # PDFs per batch for memory management

def load_keywords(excel_path):
    """Load and preprocess keywords from the stakeholder framework Excel"""
    logger.info(f"Loading keywords from {excel_path}")
    
    try:
        # Read Excel with no header to capture all data
        df = pd.read_excel(excel_path, header=None)
        logger.info(f"Successfully read Excel file with {len(df)} rows")
        
        # Extract metrics from column 3 (Relevant Dashboard Content / Metrics)
        metrics = []
        for idx, cell in enumerate(df[3]):
            if pd.notna(cell):
                cell_str = str(cell).strip()
                
                # Skip section headers and separators
                if cell_str in ['-', '|', '---', 'Relevant Dashboard Content / Metrics to Include', 'Extract exact wordings from the annual report']:
                    continue
                
                # Clean the metric: remove dashes, extra spaces, and trailing periods
                clean_metric = re.sub(r'^[-|]\s*', '', cell_str).strip()
                clean_metric = re.sub(r'\s*\.$', '', clean_metric)
                
                # Skip empty or placeholder text
                if clean_metric and len(clean_metric) > 3:
                    metrics.append(clean_metric)
        
        logger.info(f"Extracted {len(metrics)} unique metrics from Excel")
        
        # Create keyword mappings with normalization
        keyword_map = defaultdict(list)
        term_to_main = {}
        
        for metric in metrics:
            # Normalize for matching (lowercase, remove punctuation)
            normalized = re.sub(r'[^\w\s]', '', metric.lower()).strip()
            
            if normalized:
                keyword_map[normalized].append(metric)
                term_to_main[normalized] = metric
        
        # Create optimized regex pattern - sort by length to match longest terms first
        all_terms = sorted(keyword_map.keys(), key=len, reverse=True)
        logger.info(f"Created regex pattern with {len(all_terms)} terms")
        
        # Escape terms and create pattern that matches whole words
        pattern_str = r'\b(' + '|'.join(re.escape(term) for term in all_terms) + r')\b'
        return re.compile(pattern_str, re.IGNORECASE), keyword_map, term_to_main
    
    except Exception as e:
        logger.error(f"Error loading keywords: {str(e)}")
        raise

def init_worker(keyword_pattern, keyword_map, term_to_main):
    """Initialize worker processes with NLP model and keyword data"""
    global nlp, WORKER_PATTERN, WORKER_KEYWORD_MAP, WORKER_TERM_TO_MAIN
    
    try:
        # Load spaCy model once per worker
        nlp = spacy.load(NLP_MODEL, disable=["parser", "ner", "textcat"])
        nlp.add_pipe("sentencizer")
        
        WORKER_PATTERN = keyword_pattern
        WORKER_KEYWORD_MAP = keyword_map
        WORKER_TERM_TO_MAIN = term_to_main
        
        logger.info(f"Worker initialized with {len(WORKER_KEYWORD_MAP)} keywords")
    except Exception as e:
        logger.error(f"Worker initialization failed: {str(e)}")
        raise

def extract_text_with_ocr(page):
    """Extract text from page with fallback to OCR for scanned documents"""
    try:
        # First attempt: text extraction
        text = page.get_text("text", flags=fitz.TEXT_PRESERVE_LIGATURES).strip()
        
        # Second attempt: OCR if text extraction fails
        if not text or len(text) < 50:  # If text is very short, likely scanning issue
            pix = page.get_pixmap(dpi=300)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            text = pytesseract.image_to_string(img, config='--psm 1')
            logger.debug("Used OCR for page extraction")
        
        # Clean and normalize whitespace
        return re.sub(r'\s+', ' ', text).strip()
    
    except Exception as e:
        logger.warning(f"Text extraction error: {str(e)}")
        return ""

def is_valid_context(sentence, match_term, main_keyword):
    """Validate contextual relevance using NLP features and domain knowledge"""
    # Skip very short/long sentences
    if not (MIN_SENTENCE_LENGTH <= len(sentence) <= MAX_SENTENCE_LENGTH):
        return False
    
    # Convert to lowercase for case-insensitive matching
    sentence_lower = sentence.lower()
    match_term_lower = match_term.lower()
    
    # Check for common financial/digital context indicators
    financial_indicators = {
        'report', 'increase', 'decrease', 'grow', 'decline', 
        'recognize', 'comply', 'violate', 'covenant', 'obligation',
        'percent', '%', 'million', 'billion', '$', '€', '£', '¥',
        'digital', 'technology', 'online', 'internet', 'e-', 'electronic',
        'data', 'system', 'platform', 'service', 'solution', 'application',
        'shariah', 'islamic', 'halal', 'zakat', 'fatwa', 'compliant'
    }
    
    # Check if the sentence contains financial/digital context indicators
    has_context = False
    for indicator in financial_indicators:
        if indicator in sentence_lower:
            has_context = True
            break
    
    # If no clear context indicators, check proximity to numbers
    if not has_context:
        # Look for numbers near the keyword
        keyword_pos = sentence_lower.find(match_term_lower)
        if keyword_pos >= 0:
            context_window = sentence_lower[max(0, keyword_pos-20):min(len(sentence_lower), keyword_pos+20)]
            if re.search(r'\d+', context_window):
                has_context = True
    
    return has_context

def process_page(pdf_path, page_num, page):
    """Process single page for keyword extraction"""
    results = []
    try:
        text = extract_text_with_ocr(page)
        if not text or len(text) < MIN_SENTENCE_LENGTH:
            return results
        
        # Skip pages without potential matches (fast pre-scan)
        if not WORKER_PATTERN.search(text):
            return results
        
        # Process with spaCy for sentence segmentation
        doc = nlp(text)
        sentences = [sent.text for sent in doc.sents]
        
        # Process each sentence
        for sentence in sentences:
            sentence_clean = re.sub(r'[^\w\s]', '', sentence.lower()).strip()
            
            if not sentence_clean:
                continue
                
            # Find all keyword matches in sentence
            for match in WORKER_PATTERN.finditer(sentence_clean):
                match_term = match.group(1)
                
                if match_term not in WORKER_TERM_TO_MAIN:
                    continue
                
                # Get the original metric name
                main_metric = WORKER_TERM_TO_MAIN[match_term]
                
                # Validate context
                if is_valid_context(sentence, match_term, main_metric):
                    results.append({
                        'file_name': os.path.basename(pdf_path),
                        'page_number': page_num + 1,
                        'keyword': main_metric,
                        'sentence': sentence.strip()
                    })
        
        return results
    
    except Exception as e:
        logger.error(f"Page processing error in {pdf_path} page {page_num+1}: {str(e)}")
        return []

def process_pdf(pdf_path, total_pages=None):
    """Process entire PDF document"""
    start_time = time.time()
    results = []
    
    try:
        doc = fitz.open(pdf_path)
        if total_pages is None:
            total_pages = min(len(doc), 300)  # Limit to 300 pages to avoid extremely large reports
        
        logger.info(f"Processing {pdf_path} ({total_pages} pages)")
        
        # Process pages sequentially (better memory management for large PDFs)
        for page_num in range(total_pages):
            try:
                page = doc.load_page(page_num)
                page_results = process_page(pdf_path, page_num, page)
                results.extend(page_results)
            except Exception as e:
                logger.warning(f"Error processing page {page_num+1} in {pdf_path}: {str(e)}")
                continue
        
        doc.close()
        processing_time = time.time() - start_time
        logger.info(f"Processed {pdf_path} ({len(results)} matches) in {processing_time:.2f}s")
        return results
    
    except Exception as e:
        logger.error(f"PDF processing error for {pdf_path}: {str(e)}")
        return []

def main():
    # Configuration
    KEYWORD_EXCEL = "C:/Users/Dell/Documents/extract.xlsx"
    PDF_DIR = "C:/Users/Dell/Documents/cleaned_Annual_Report"
    OUTPUT_FILE = "extracted_results.xlsx"
    
    # Validate input directory
    if not os.path.exists(PDF_DIR):
        logger.error(f"PDF directory not found: {PDF_DIR}")
        raise FileNotFoundError(f"PDF directory not found: {PDF_DIR}")
    
    # Load keywords
    try:
        keyword_pattern, keyword_map, term_to_main = load_keywords(KEYWORD_EXCEL)
        logger.info(f"Successfully loaded {len(keyword_map)} keywords with variations")
    except Exception as e:
        logger.error(f"Failed to load keywords: {str(e)}")
        raise
    
    # Get PDF files
    pdf_files = [
        os.path.join(PDF_DIR, f) 
        for f in os.listdir(PDF_DIR) 
        if f.lower().endswith('.pdf') and not f.startswith('.')
    ]
    
    if not pdf_files:
        logger.error(f"No PDF files found in {PDF_DIR}")
        raise ValueError(f"No PDF files found in {PDF_DIR}")
    
    logger.info(f"Found {len(pdf_files)} PDFs to process")
    
    # Process in batches to manage memory
    all_results = []
    total_files = len(pdf_files)
    
    for i in range(0, total_files, BATCH_SIZE):
        batch = pdf_files[i:i+BATCH_SIZE]
        batch_num = (i // BATCH_SIZE) + 1
        total_batches = (total_files + BATCH_SIZE - 1) // BATCH_SIZE
        
        logger.info(f"Processing batch {batch_num} of {total_batches} ({len(batch)} files)")
        
        with ProcessPoolExecutor(
            max_workers=max(1, min(4, os.cpu_count() or 4)),
            initializer=init_worker,
            initargs=(keyword_pattern, keyword_map, term_to_main)
        ) as executor:
            futures = {executor.submit(process_pdf, pdf): pdf for pdf in batch}
            
            # Use tqdm for progress bar
            with tqdm(total=len(batch), desc=f"Batch {batch_num}") as pbar:
                for future in as_completed(futures):
                    pdf = futures[future]
                    try:
                        results = future.result()
                        all_results.extend(results)
                    except Exception as e:
                        logger.exception(f"Error processing {pdf}")
                    finally:
                        pbar.update(1)
        
        # Memory cleanup between batches
        gc.collect()
    
    # Create and clean results
    if all_results:
        df = pd.DataFrame(all_results)
        
        # Deduplicate while preserving first occurrence
        df = df.drop_duplicates(subset=['file_name', 'page_number', 'keyword', 'sentence'])
        
        # Sort for readability
        df = df.sort_values(['keyword', 'file_name', 'page_number'])
        
        # Add stakeholder context if available
        if 'stakeholder_group' in df.columns:
            df = df.sort_values(['stakeholder_group', 'keyword', 'file_name', 'page_number'])
        
        # Export results
        df.to_excel(OUTPUT_FILE, index=False)
        logger.info(f"Extraction complete. {len(df)} records saved to {OUTPUT_FILE}")
        
        # Create summary report
        summary = df['keyword'].value_counts().reset_index()
        summary.columns = ['Keyword', 'Occurrences']
        summary_file = "extraction_summary.xlsx"
        summary.to_excel(summary_file, index=False)
        logger.info(f"Summary report saved to {summary_file}")
        
        return df
    else:
        logger.warning("No results extracted. Check logs for potential issues.")
        return pd.DataFrame(columns=['file_name', 'page_number', 'keyword', 'sentence'])

if __name__ == "__main__":
    start = time.time()
    try:
        logger.info("Starting annual report extraction process")
        result_df = main()
        
        if not result_df.empty:
            logger.info(f"Successfully extracted {len(result_df)} relevant statements")
        else:
            logger.warning("No relevant information was extracted from the documents")
            
    except Exception as e:
        logger.exception("Critical error in extraction process")
        raise
    finally:
        elapsed = time.time() - start
        logger.info(f"Total execution time: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

Batch 3: 100%|██████████| 45/45 [00:01<00:00, 40.67it/s]


In [16]:
import multiprocessing  # Add this line
import os
import re
import logging
import fitz  # PyMuPDF
import pandas as pd
import numpy as np
from PIL import Image
import pytesseract
from collections import defaultdict
import time
from tqdm import tqdm
import gc
import sys

# Configure logging
logging.basicConfig(
    filename='extraction.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# Global constants
MIN_SENTENCE_LENGTH = 15  # Filter out fragments
MAX_SENTENCE_LENGTH = 500  # Prevent paragraph captures
BATCH_SIZE = 10  # Reduced batch size for Windows memory constraints
MAX_PAGES = 300  # Limit pages per PDF to prevent memory issues

def load_keywords(excel_path):
    """Load and preprocess keywords from the stakeholder framework Excel"""
    logger.info(f"Loading keywords from {excel_path}")
    
    try:
        # Read Excel with no header to capture all data
        df = pd.read_excel(excel_path, header=None)
        logger.info(f"Successfully read Excel file with {len(df)} rows")
        
        # Extract metrics from column 3 (Relevant Dashboard Content / Metrics)
        metrics = []
        for idx, cell in enumerate(df[3]):
            if pd.notna(cell):
                cell_str = str(cell).strip()
                
                # Skip section headers and separators
                if cell_str in ['-', '|', '---', 'Relevant Dashboard Content / Metrics to Include', 'Extract exact wordings from the annual report']:
                    continue
                
                # Clean the metric: remove dashes, extra spaces, and trailing periods
                clean_metric = re.sub(r'^[-|]\s*', '', cell_str).strip()
                clean_metric = re.sub(r'\s*\.$', '', clean_metric)
                
                # Skip empty or placeholder text
                if clean_metric and len(clean_metric) > 3:
                    metrics.append(clean_metric)
        
        logger.info(f"Extracted {len(metrics)} unique metrics from Excel")
        
        # Create keyword mappings with normalization
        keyword_map = defaultdict(list)
        term_to_main = {}
        
        for metric in metrics:
            # Normalize for matching (lowercase, remove punctuation)
            normalized = re.sub(r'[^\w\s]', '', metric.lower()).strip()
            
            if normalized:
                keyword_map[normalized].append(metric)
                term_to_main[normalized] = metric
        
        # Create optimized regex pattern - sort by length to match longest terms first
        all_terms = sorted(keyword_map.keys(), key=len, reverse=True)
        logger.info(f"Created regex pattern with {len(all_terms)} terms")
        
        # Escape terms and create pattern that matches whole words
        pattern_str = r'\b(' + '|'.join(re.escape(term) for term in all_terms) + r')\b'
        return re.compile(pattern_str, re.IGNORECASE), keyword_map, term_to_main
    
    except Exception as e:
        logger.error(f"Error loading keywords: {str(e)}")
        raise

def extract_text_with_ocr(page):
    """Extract text from page with fallback to OCR for scanned documents"""
    try:
        # First attempt: text extraction
        text = page.get_text("text", flags=fitz.TEXT_PRESERVE_LIGATURES).strip()
        
        # Second attempt: OCR if text extraction fails
        if not text or len(text) < 50:  # If text is very short, likely scanning issue
            pix = page.get_pixmap(dpi=300)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            text = pytesseract.image_to_string(img, config='--psm 1')
            logger.debug("Used OCR for page extraction")
        
        # Clean and normalize whitespace
        return re.sub(r'\s+', ' ', text).strip()
    
    except Exception as e:
        logger.warning(f"Text extraction error: {str(e)}")
        return ""

def is_valid_context(sentence, match_term, main_keyword):
    """Validate contextual relevance using basic string operations (no spaCy)"""
    # Skip very short/long sentences
    if not (MIN_SENTENCE_LENGTH <= len(sentence) <= MAX_SENTENCE_LENGTH):
        return False
    
    # Convert to lowercase for case-insensitive matching
    sentence_lower = sentence.lower()
    match_term_lower = match_term.lower()
    
    # Check for common financial/digital context indicators
    financial_indicators = {
        'report', 'increase', 'decrease', 'grow', 'decline', 
        'recognize', 'comply', 'violate', 'covenant', 'obligation',
        'percent', '%', 'million', 'billion', '$', '€', '£', '¥',
        'digital', 'technology', 'online', 'internet', 'e-', 'electronic',
        'data', 'system', 'platform', 'service', 'solution', 'application',
        'shariah', 'islamic', 'halal', 'zakat', 'fatwa', 'compliant'
    }
    
    # Check if the sentence contains financial/digital context indicators
    has_context = False
    for indicator in financial_indicators:
        if indicator in sentence_lower:
            has_context = True
            break
    
    # If no clear context indicators, check proximity to numbers
    if not has_context:
        # Look for numbers near the keyword
        keyword_pos = sentence_lower.find(match_term_lower)
        if keyword_pos >= 0:
            context_window = sentence_lower[max(0, keyword_pos-20):min(len(sentence_lower), keyword_pos+20)]
            if re.search(r'\d+', context_window):
                has_context = True
    
    return has_context

def process_page(pdf_path, page_num, page, keyword_pattern, keyword_map, term_to_main):
    """Process single page for keyword extraction (no spaCy dependency)"""
    results = []
    try:
        text = extract_text_with_ocr(page)
        if not text or len(text) < MIN_SENTENCE_LENGTH:
            return results
        
        # Skip pages without potential matches (fast pre-scan)
        if not keyword_pattern.search(text):
            return results
        
        # Simple sentence splitting (replacing spaCy)
        sentences = re.split(r'(?<=[.!?])\s+', text)
        
        # Process each sentence
        for sentence in sentences:
            sentence_clean = re.sub(r'[^\w\s]', '', sentence.lower()).strip()
            
            if not sentence_clean:
                continue
                
            # Find all keyword matches in sentence
            for match in keyword_pattern.finditer(sentence_clean):
                match_term = match.group(1)
                
                if match_term not in term_to_main:
                    continue
                
                # Get the original metric name
                main_metric = term_to_main[match_term]
                
                # Validate context
                if is_valid_context(sentence, match_term, main_metric):
                    results.append({
                        'file_name': os.path.basename(pdf_path),
                        'page_number': page_num + 1,
                        'keyword': main_metric,
                        'sentence': sentence.strip()
                    })
        
        return results
    
    except Exception as e:
        logger.error(f"Page processing error in {pdf_path} page {page_num+1}: {str(e)}")
        return []

def process_pdf(pdf_path, keyword_pattern, keyword_map, term_to_main, total_pages=None):
    """Process entire PDF document"""
    start_time = time.time()
    results = []
    
    try:
        doc = fitz.open(pdf_path)
        if total_pages is None:
            total_pages = min(len(doc), MAX_PAGES)  # Limit to prevent memory issues
        
        logger.info(f"Processing {pdf_path} ({total_pages} pages)")
        
        # Process pages sequentially (critical for Windows)
        for page_num in range(total_pages):
            try:
                page = doc.load_page(page_num)
                page_results = process_page(pdf_path, page_num, page, 
                                          keyword_pattern, keyword_map, term_to_main)
                results.extend(page_results)
            except Exception as e:
                logger.warning(f"Error processing page {page_num+1} in {pdf_path}: {str(e)}")
                continue
        
        doc.close()
        processing_time = time.time() - start_time
        logger.info(f"Processed {pdf_path} ({len(results)} matches) in {processing_time:.2f}s")
        return results
    
    except Exception as e:
        logger.error(f"PDF processing error for {pdf_path}: {str(e)}")
        return []

def main():
    # Configuration
    KEYWORD_EXCEL = "C:/Users/Dell/Documents/extract.xlsx"
    PDF_DIR = "C:/Users/Dell/Documents/cleaned_Annual_Report"
    OUTPUT_FILE = "extracted_results.xlsx"
    
    # Validate input directory
    if not os.path.exists(PDF_DIR):
        logger.error(f"PDF directory not found: {PDF_DIR}")
        raise FileNotFoundError(f"PDF directory not found: {PDF_DIR}")
    
    # Load keywords
    try:
        keyword_pattern, keyword_map, term_to_main = load_keywords(KEYWORD_EXCEL)
        logger.info(f"Successfully loaded {len(keyword_map)} keywords with variations")
    except Exception as e:
        logger.error(f"Failed to load keywords: {str(e)}")
        raise
    
    # Get PDF files
    pdf_files = [
        os.path.join(PDF_DIR, f) 
        for f in os.listdir(PDF_DIR) 
        if f.lower().endswith('.pdf') and not f.startswith('.')
    ]
    
    if not pdf_files:
        logger.error(f"No PDF files found in {PDF_DIR}")
        raise ValueError(f"No PDF files found in {PDF_DIR}")
    
    logger.info(f"Found {len(pdf_files)} PDFs to process")
    
    # Process in batches to manage memory
    all_results = []
    total_files = len(pdf_files)
    
    # Process sequentially on Windows (no multiprocessing)
    logger.info("Starting sequential processing (Windows-compatible mode)")
    
    for i, pdf_path in enumerate(tqdm(pdf_files, desc="Processing PDFs")):
        try:
            results = process_pdf(
                pdf_path, 
                keyword_pattern, 
                keyword_map, 
                term_to_main
            )
            all_results.extend(results)
        except Exception as e:
            logger.error(f"Critical error processing {pdf_path}: {str(e)}")
        
        # Memory cleanup between files
        if (i + 1) % 10 == 0 or i == len(pdf_files) - 1:
            gc.collect()
    
    # Create and clean results
    if all_results:
        df = pd.DataFrame(all_results)
        
        # Deduplicate while preserving first occurrence
        df = df.drop_duplicates(subset=['file_name', 'page_number', 'keyword', 'sentence'])
        
        # Sort for readability
        df = df.sort_values(['keyword', 'file_name', 'page_number'])
        
        # Export results
        df.to_excel(OUTPUT_FILE, index=False)
        logger.info(f"Extraction complete. {len(df)} records saved to {OUTPUT_FILE}")
        
        # Create summary report
        summary = df['keyword'].value_counts().reset_index()
        summary.columns = ['Keyword', 'Occurrences']
        summary_file = "extraction_summary.xlsx"
        summary.to_excel(summary_file, index=False)
        logger.info(f"Summary report saved to {summary_file}")
        
        return df
    else:
        logger.warning("No results extracted. Check logs for potential issues.")
        return pd.DataFrame(columns=['file_name', 'page_number', 'keyword', 'sentence'])

if __name__ == "__main__":
    # Windows multiprocessing fix
    if sys.platform == "win32":
        # Required for Windows multiprocessing
        multiprocessing.freeze_support()
    
    start = time.time()
    try:
        logger.info("Starting annual report extraction process")
        result_df = main()
        
        if not result_df.empty:
            logger.info(f"Successfully extracted {len(result_df)} relevant statements")
        else:
            logger.warning("No relevant information was extracted from the documents")
            
    except Exception as e:
        logger.exception("Critical error in extraction process")
        raise
    finally:
        elapsed = time.time() - start
        logger.info(f"Total execution time: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

Processing PDFs: 100%|██████████| 145/145 [04:48<00:00,  1.99s/it]
